# 02 - Baseline Models And Submission

This notebook builds reproducible starter baselines.

Modeling choices here are deliberately conservative:
- exclude `ticker` from the first baseline
- use only features available at the observation date
- evaluate on a 2022 holdout before fitting on all training rows

The goal is not leaderboard magic. The goal is a clean baseline that the team can improve.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', 100)

ROOT = Path.cwd().resolve().parent
DATA_DIR = ROOT / 'data' / 'raw'
SUBMISSION_DIR = ROOT / 'submissions'
SUBMISSION_DIR.mkdir(exist_ok=True)

sys.path.insert(0, str(ROOT / 'scripts'))
from data_prep import (  # noqa: E402
    load_raw,
    build_features,
    time_split,
    clip_target_train_only,
    rmse,
)

train, test, sample_submission = load_raw(DATA_DIR)
train.shape, test.shape

((23070, 39), (8520, 36))

In [2]:
# `build_features` produces the same column layout for train and test:
#   - drops id, ticker, period_start, period_end, return_pct
#   - keeps start_year, adds `years_since_2019`
#   - adds `<col>_is_missing` flags for the high-missingness fields
# It does NOT impute, scale, or clip — those happen inside the pipeline below
# so they can be fit on the train fold only.
X_full = build_features(train)
X_test = build_features(test)
y_full = train['return_pct']

assert list(X_full.columns) == list(X_test.columns), (
    'train/test feature columns must match exactly'
)
print('Number of modeling features:', X_full.shape[1])
display(X_full.head())

Number of modeling features: 50


,start_year,pe_ttm,price_to_book,price_to_sales,growth_pe_ratio,gross_margin,operating_margin,net_margin,roa,roe,rote,revenue_growth_3y,revenue_growth_yoy,revenue_ttm,net_income_ttm,income_before_tax,eps_basic,eps_diluted,total_assets,stockholders_equity,current_assets,current_liabilities,long_term_debt,goodwill,inventory,current_ratio,quick_ratio,debt_to_equity,dividend_yield,dividends_ttm,dividends_paid_ttm,shares_outstanding,shares_diluted,sector_code,years_since_2019,dividends_paid_ttm_is_missing,dividend_yield_is_missing,dividends_ttm_is_missing,gross_margin_is_missing,inventory_is_missing,debt_to_equity_is_missing,growth_pe_ratio_is_missing,long_term_debt_is_missing,shares_outstanding_is_missing,goodwill_is_missing,quick_ratio_is_missing,current_ratio_is_missing,revenue_growth_3y_is_missing,current_liabilities_is_missing,current_assets_is_missing
0,2022.0,25.28,38.23,6.68,1.95,43.75,30.82,26.41,29.07,151.24,151.24,49.34,18.63,3.860170e+11,1.019350e+11,3.013900e+10,1.54,1.52,3.506620e+11,6.739900e+10,1.181800e+11,1.275080e+11,2.066460e+11,0.0,5.460000e+09,0.93,0.88,3.07,0.57,1.473400e+10,1.468700e+10,1.620757e+10,1.640332e+10,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2022.0,20.97,35.95,5.39,2.37,43.26,27.82,25.71,29.63,171.46,171.46,49.61,11.63,3.875420e+11,9.963300e+10,2.306600e+10,1.20,1.20,3.363090e+11,5.810700e+10,1.122920e+11,1.298730e+11,1.894000e+11,0.0,5.433000e+09,0.86,0.82,3.26,0.71,1.477800e+10,1.473400e+10,1.609538e+10,1.626220e+10,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2022.0,22.23,43.78,5.63,2.32,43.31,30.29,25.31,28.29,196.96,196.96,51.56,7.79,3.943280e+11,9.980300e+10,1.191030e+11,6.15,6.11,3.527550e+11,5.067200e+10,1.354050e+11,1.539820e+11,1.979180e+11,0.0,4.946000e+09,0.88,0.85,3.91,0.67,1.484100e+10,1.479300e+10,1.594342e+10,1.632582e+10,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2022.0,20.13,33.78,4.94,2.22,42.96,30.74,24.56,27.45,167.77,167.77,44.77,2.44,3.875370e+11,9.517100e+10,3.562300e+10,1.89,1.88,3.467470e+11,5.672700e+10,1.287770e+11,1.372860e+11,1.992540e+11,0.0,6.820000e+09,0.94,0.89,3.51,0.78,1.487700e+10,1.484000e+10,1.584241e+10,1.595572e+10,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2021.0,23.43,25.84,5.49,1.35,42.51,30.70,23.45,22.63,110.31,110.31,31.52,21.43,3.254060e+11,7.631100e+10,2.801100e+10,1.41,1.40,3.371580e+11,6.917800e+10,1.214650e+11,1.063850e+11,2.172840e+11,0.0,5.219000e+09,1.14,1.09,3.14,0.80,1.422700e+10,1.421200e+10,1.668630e+10,1.692916e+10,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
# Time-based split. The 2022 holdout's forward window (2023) is the closest
# analog we have to the test situation, where the forward window (2024-2025)
# lies entirely outside the training label coverage.
split = time_split(train, valid_year=2022)
print(split.description)

X_train = X_full.iloc[split.train_idx]
y_train_raw = y_full.iloc[split.train_idx]
X_valid = X_full.iloc[split.valid_idx]
y_valid = y_full.iloc[split.valid_idx]

# Clip the *training* target only — never the validation target. Cutoffs are
# fit on the train fold so no future information leaks back.
y_train, lo, hi = clip_target_train_only(y_train_raw, lower_pct=1.0, upper_pct=99.0)
print(f'training target clipped at lo={lo:.2f}, hi={hi:.2f} '
      f'(std before={y_train_raw.std():.2f}, after={y_train.std():.2f})')
print('X_train shape:', X_train.shape)
print('X_valid shape:', X_valid.shape)

train start_year<2022 -> validate start_year==2022; validation forward window ends 2023-12-31
training target clipped at lo=-81.01, hi=327.94 (std before=158.96, after=64.32)
X_train shape: (16436, 50)
X_valid shape: (6634, 50)


In [4]:
ridge_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', Ridge(alpha=1.0)),
    ]
)

hgb_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        (
            'model',
            HistGradientBoostingRegressor(
                learning_rate=0.05,
                max_depth=4,
                max_iter=300,
                min_samples_leaf=40,
                random_state=42,
            ),
        ),
    ]
)

models = {
    'dummy_mean': DummyRegressor(strategy='mean'),
    'dummy_median': DummyRegressor(strategy='median'),
    'ridge': ridge_pipeline,
    'hist_gradient_boosting': hgb_pipeline,
}

scores = []
fitted_models = {}

for name, model in models.items():
    # Fit on the clipped training target; score against the raw validation target.
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    score = rmse(y_valid, preds)
    scores.append({'model': name, 'rmse': score})
    fitted_models[name] = model

scores = pd.DataFrame(scores).sort_values('rmse')
display(scores)

,model,rmse
0,dummy_mean,64.880177
1,dummy_median,65.350859
3,hist_gradient_boosting,68.424683
2,ridge,86.426329


In [5]:
best_model_name = scores.iloc[0]['model']
best_model = fitted_models[best_model_name]

valid_predictions = best_model.predict(X_valid)
residuals = pd.DataFrame(
    {
        'ticker': train.iloc[split.valid_idx]['ticker'].values,
        'period_start': train.iloc[split.valid_idx]['period_start'].values,
        'actual': y_valid.values,
        'predicted': valid_predictions,
    }
)
residuals['abs_error'] = (residuals['actual'] - residuals['predicted']).abs()

print('Best validation model:', best_model_name)
display(residuals.sort_values('abs_error', ascending=False).head(15))

Best validation model: dummy_mean


,ticker,period_start,actual,predicted,abs_error
1721,SLNO,2022-12-31,1932.83,16.190014,1916.639986
1720,SLNO,2022-09-30,1667.07,16.190014,1650.879986
6287,ACIC,2022-09-30,1050.00,16.190014,1033.809986
1198,CVNA,2022-12-31,1016.88,16.190014,1000.689986
5139,AAOI,2022-12-31,922.22,16.190014,906.029986
6288,ACIC,2022-12-31,792.45,16.190014,776.259986
4270,CIFR,2022-12-31,637.50,16.190014,621.309986
3178,IMVT,2022-09-30,587.99,16.190014,571.799986
2170,MARA,2022-12-31,586.84,16.190014,570.649986
5631,AVDL,2022-06-30,575.41,16.190014,559.219986


## Refit On All Training Data

After selecting a baseline model using the holdout, refit it on the full training set before generating the Kaggle submission.

In [6]:
# Refit on ALL training data (including 2022) with the same target-clipping
# cutoffs re-estimated from the full training target. The most recent year
# carries the most information about the regime closest to the 2024 test window.
y_full_clipped, lo_full, hi_full = clip_target_train_only(y_full, 1.0, 99.0)
print(f'final-fit target clipped at lo={lo_full:.2f}, hi={hi_full:.2f}')

final_model = models[best_model_name]
final_model.fit(X_full, y_full_clipped)
test_predictions = final_model.predict(X_test)

submission = sample_submission.copy()
submission['return_pct'] = test_predictions

submission_path = SUBMISSION_DIR / f'{best_model_name}_baseline.csv'
submission.to_csv(submission_path, index=False)

print('Saved submission to:', submission_path)
display(submission.head())
print('Submission rows:', len(submission))
assert len(submission) == len(sample_submission), 'submission row count must match sample_submission'

final-fit target clipped at lo=-80.20, hi=299.17
Saved submission to: D:\Documents\Programms\kaggle-competition-stock-return-fundamentals\submissions\dummy_mean_baseline.csv


,id,return_pct
0,0,14.461954
1,1,14.461954
2,2,14.461954
3,3,14.461954
4,4,14.461954


Submission rows: 8520


## Next Experiments

Once this notebook runs cleanly, the next sensible experiments are:

- target clipping or winsorization on the training fold only
- sector-relative features
- log transforms for scale-heavy accounting variables
- LightGBM or XGBoost with the same time-based validation split
- model blending once two models show independent value